# Evaluate Generated Samples

Load chunked samples from the `samples/` directory, compute metrics against real data, and visualize.

In [ ]:
import datetime
from pathlib import Path

import numpy as np
import torch

from cfm.data.loading import build_cfm_pairs, compute_daily_rv, load_rv
from cfm.evaluation.metrics import evaluate_all
from cfm.evaluation.visualize import (
    plot_acf,
    plot_diurnal_pattern,
    plot_marginal_distributions,
    plot_sample_paths,
)

# --- Config ---
SAMPLES_DIR = Path("../samples/seed_42/test")
HARXHAR_PATH = "../data/all30min"
CONTEXT_DAYS = 5
TRAIN_END = "2020-12-31"
VAL_END = "2022-12-31"

In [ ]:
# Load all chunks
chunks = sorted(SAMPLES_DIR.glob("chunk_*.pt"))
print(f"Found {len(chunks)} chunks")

data = [torch.load(c, weights_only=False) for c in chunks]
gen_proportions = torch.cat([d["generated_proportions"] for d in data]).numpy()
gen_absolute = torch.cat([d["generated_absolute"] for d in data]).numpy()
dates = np.concatenate([d["dates"] for d in data])
print(f"Total samples: {len(gen_proportions)}, date range: {dates.min()} to {dates.max()}")

In [ ]:
# Load real data for comparison
rv_df = load_rv(HARXHAR_PATH)
daily_df = compute_daily_rv(rv_df)
conditions_raw, proportions_raw, dates_raw = build_cfm_pairs(daily_df, CONTEXT_DAYS)

val_end_dt = datetime.date.fromisoformat(VAL_END)
test_mask = dates_raw > np.datetime64(val_end_dt)
real_proportions = proportions_raw[test_mask]
test_daily_rv = conditions_raw[test_mask, 0] ** 2
print(f"Real test samples: {len(real_proportions)}")

In [ ]:
# Compute metrics
metrics = evaluate_all(real_proportions, gen_proportions, daily_rv=test_daily_rv)
for k, v in metrics.items():
    print(f"  {k:30s} {v:.6f}")

In [ ]:
# Sample paths
fig, ax = plot_sample_paths(real_proportions, gen_proportions)
fig.tight_layout()

In [ ]:
# Diurnal pattern
fig, ax = plot_diurnal_pattern(real_proportions, gen_proportions)
fig.tight_layout()

In [ ]:
# Marginal distributions
fig, axes = plot_marginal_distributions(real_proportions, gen_proportions)
fig.tight_layout()

In [ ]:
# ACF comparison
from cfm.evaluation.metrics import acf_comparison

acf = acf_comparison(real_proportions, gen_proportions)
fig, ax = plot_acf(acf["real_acf"], acf["gen_acf"])
fig.tight_layout()